# Day 13 / 42: Linear Regression
### 42 Days of ML Challenge | @VaishnaviJagtap18

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/VaishnaviJagtap18/42-days-aiml-challenge/blob/main/week2_feature_work/day13_linear_regression/day13_notebook.ipynb)

---

## What You Will Learn
- Build a multi-feature linear regression model on house price data
- R², RMSE, and MAE: what each metric actually tells you, in plain terms
- Reading coefficients correctly: what "price increases by X per sqft" really means
- Multicollinearity: the bug that makes coefficients lie to you
- Why Zillow still uses linear regression as a baseline in 2025

---

## Step 0: Install and Import

In [ ]:
!pip install numpy pandas matplotlib scikit-learn --quiet

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
print("All imports successful. You are ready for Day 13.")

---
## Step 1: The Dataset — House Prices With 3 Features

Yesterday (Day 12) we used 1 feature (sqft) to keep the math visible.

Today we use 3 features, which is what linear regression looks like in practice:

$$\hat{y} = w_1 \cdot \text{sqft} + w_2 \cdot \text{bedrooms} + w_3 \cdot \text{age} + b$$

Each feature gets its own weight. The model learns all of them at once.

In [ ]:
np.random.seed(42)
n = 300

sqft     = np.random.normal(1500, 400, n)
bedrooms = np.random.randint(1, 5, n).astype(float)
age      = np.random.randint(0, 30, n).astype(float)

# True relationship: bigger house = more expensive, more bedrooms = more expensive,
# older house = cheaper. Plus random noise (real-world variation).
price = 50 + 0.12*sqft + 15*bedrooms - 1.2*age + np.random.normal(0, 20, n)

df = pd.DataFrame({
    'sqft': sqft,
    'bedrooms': bedrooms,
    'age_years': age,
    'price_lakhs': price
})

print("Sample data:")
print(df.head(8).round(1).to_string(index=False))
print(f"\nDataset size: {n} houses")
print(f"Price range: {price.min():.1f} to {price.max():.1f} lakhs")

---
## Step 2: Train the Model

Split into train/test, fit `LinearRegression()`, and look at what it learned.

In [ ]:
X = df[['sqft', 'bedrooms', 'age_years']].values
y = df['price_lakhs'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

feature_names = ['sqft', 'bedrooms', 'age_years']
print("=== What the model learned ===")
print(f"Intercept (b): {model.intercept_:.4f}")
print()
for name, coef in zip(feature_names, model.coef_):
    print(f"  {name:>12}: {coef:>8.4f}")

print()
print("Reading these coefficients:")
print(f"  +1 sqft       -> price changes by {model.coef_[0]:.4f} lakhs (holding bedrooms, age constant)")
print(f"  +1 bedroom    -> price changes by {model.coef_[1]:.4f} lakhs (holding sqft, age constant)")
print(f"  +1 year older -> price changes by {model.coef_[2]:.4f} lakhs (holding sqft, bedrooms constant)")
print()
print("True values used to generate this data: sqft=0.12, bedrooms=15, age=-1.2")
print("The model recovered values very close to the truth from noisy data alone.")

---
## Step 3: R², RMSE, MAE — What Each One Actually Tells You

Three numbers get thrown around constantly. Here's what each one MEANS, not just the formula.

**R² (R-squared)**: "What fraction of the variation in price does the model explain?"
- Range: 0 to 1 (can go negative for a model worse than just guessing the average)
- R² = 0.77 means 77% of the variation in house prices is explained by sqft, bedrooms, and age
- The remaining 23% is noise, or factors not in the model (location, condition, etc.)

**RMSE (Root Mean Squared Error)**: "On average, how far off are predictions, in the SAME units as the target?"
- RMSE = 21.5 means predictions are typically off by about 21.5 lakhs
- Squares errors first, so large mistakes are punished more

**MAE (Mean Absolute Error)**: "On average, how far off are predictions?" — but without squaring
- More intuitive, less sensitive to a few big outliers than RMSE
- If RMSE >> MAE, you likely have a few large errors (outlier predictions) pulling RMSE up

In [ ]:
r2   = r2_score(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred) ** 0.5
mae  = mean_absolute_error(y_test, y_pred)

print("=== Model Performance on Test Set ===")
print(f"  R²:   {r2:.4f}   ->  Model explains {r2*100:.1f}% of price variation")
print(f"  RMSE: {rmse:.2f} lakhs  ->  Typical error is about {rmse:.1f} lakhs")
print(f"  MAE:  {mae:.2f} lakhs  ->  Average absolute error is {mae:.1f} lakhs")
print()
print(f"RMSE ({rmse:.1f}) is higher than MAE ({mae:.1f}).")
print("This gap tells you a few predictions are off by a larger amount,")
print("pulling RMSE up more than MAE (since RMSE squares errors).")
print()

# Compare to a baseline: always predicting the mean price
baseline_pred = np.full_like(y_test, y_train.mean())
baseline_rmse = mean_squared_error(y_test, baseline_pred) ** 0.5
baseline_r2 = r2_score(y_test, baseline_pred)

print("=== Baseline: always predict the average price ===")
print(f"  Baseline RMSE: {baseline_rmse:.2f} lakhs")
print(f"  Baseline R²:   {baseline_r2:.4f}  (always 0 by definition)")
print()
print(f"Our model's RMSE ({rmse:.1f}) vs baseline RMSE ({baseline_rmse:.1f}):")
print(f"That's a {(1 - rmse/baseline_rmse)*100:.1f}% reduction in error over guessing the average.")
print("R² is just a rescaled version of this comparison.")

---
## Step 4: Visualize Predictions vs Actual

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: predicted vs actual (perfect model = points on the diagonal line)
axes[0].scatter(y_test, y_pred, alpha=0.5, color='steelblue', edgecolor='white')
lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
axes[0].plot(lims, lims, 'r--', linewidth=2, label='Perfect prediction')
axes[0].set_xlabel('Actual Price (lakhs)')
axes[0].set_ylabel('Predicted Price (lakhs)')
axes[0].set_title(f'Predicted vs Actual  (R²={r2:.3f})')
axes[0].legend()

# Right: residuals — errors should be randomly scattered around 0
residuals = y_test - y_pred
axes[1].scatter(y_pred, residuals, alpha=0.5, color='darkorange', edgecolor='white')
axes[1].axhline(0, color='red', linestyle='--', linewidth=2)
axes[1].set_xlabel('Predicted Price (lakhs)')
axes[1].set_ylabel('Residual (Actual - Predicted)')
axes[1].set_title('Residual Plot')

plt.tight_layout()
plt.savefig('day13_predictions.png', dpi=120, bbox_inches='tight')
plt.show()

print(f"Residuals mean: {residuals.mean():.2f} (should be close to 0)")
print(f"Residuals std:  {residuals.std():.2f}")
print()
print("If the residual plot shows a PATTERN (a curve, a funnel shape), it means")
print("linear regression is missing something — the relationship isn't purely linear,")
print("or an important feature is missing.")
print("Random scatter around 0, like here, means the linear model fits well.")

---
## Step 5: Multicollinearity — When Coefficients Lie

What happens if two features carry almost the same information?

**Scenario:** Someone adds a second "size" feature — say, sqft measured by two different sources, nearly identical but not exact. Both features tell the model roughly the same thing.

The model still predicts well. But the individual coefficients become unstable and meaningless on their own.

In [ ]:
# Create a near-duplicate of sqft (e.g. measured by a different agency, slightly different numbers)
sqft_duplicate = sqft + np.random.normal(0, 5, n)

print(f"Correlation between sqft and sqft_duplicate: {np.corrcoef(sqft, sqft_duplicate)[0,1]:.4f}")
print("(1.0 = perfectly correlated, these are nearly identical)")
print()

X_dup = np.column_stack([sqft, sqft_duplicate, bedrooms, age])

print("=== Coefficient for 'sqft' across 5 different train/test splits ===")
print(f"{'Split':>6} | {'sqft coef':>12} | {'sqft_dup coef':>14} | {'SUM':>10}")
print("-" * 50)
for seed in range(5):
    Xtr, Xte, ytr, yte = train_test_split(X_dup, price, test_size=0.2, random_state=seed)
    m = LinearRegression().fit(Xtr, ytr)
    c1, c2 = m.coef_[0], m.coef_[1]
    print(f"{seed:>6} | {c1:>12.4f} | {c2:>14.4f} | {c1+c2:>10.4f}")

print()
print("Notice: individual coefficients swing wildly (even sign-flip), but their SUM")
print("stays stable around 0.12 — the true combined effect of 'size' on price.")
print()
print("This is multicollinearity. The model can't tell which of the two identical")
print("features deserves the credit, so it splits it arbitrarily between them.")
print()
print("R² and predictions stay fine. But if you read individual coefficients")
print("as 'feature importance' or 'business insight', you get a different")
print("(and wrong) story every time you retrain.")

---
## Step 6: The Real-World Production Problem

**The scenario:** A real estate analytics team builds a linear regression model to explain price drivers. They present to stakeholders: "square footage measured by Source A adds 0.13 lakhs per unit, while square footage measured by Source B SUBTRACTS 0.02 lakhs per unit. Source B size is apparently bad for price."

This conclusion gets used to justify a business decision.

**What actually happened:** Source A and Source B are two slightly different measurements of the SAME thing (multicollinearity). The coefficient split between them is arbitrary — re-run the model with a different random seed and the story flips entirely, as shown above.

**The fix:**
- Check correlation between features before interpreting coefficients (correlation > 0.8-0.9 is a red flag)
- Drop or combine highly correlated features before drawing conclusions from coefficients
- If you need both features for prediction accuracy but not for interpretation, that's fine — just don't read individual coefficients as standalone "importance"
- For pure prediction (not interpretation), multicollinearity often doesn't hurt accuracy at all — R² stayed ~0.77 in both cases above

**Rule:** Linear regression coefficients are only meaningful as "effect size" when features are not strongly correlated with each other.

In [ ]:
# A quick correlation check you should run BEFORE trusting any coefficient
feature_df = pd.DataFrame({
    'sqft': sqft,
    'sqft_duplicate': sqft_duplicate,
    'bedrooms': bedrooms,
    'age': age
})

corr_matrix = feature_df.corr()
print("=== Feature Correlation Matrix ===")
print(corr_matrix.round(3))
print()

# Flag any pair above 0.8
print("=== Multicollinearity Check (threshold: 0.8) ===")
flagged = False
for i, col1 in enumerate(corr_matrix.columns):
    for j, col2 in enumerate(corr_matrix.columns):
        if i < j and abs(corr_matrix.iloc[i, j]) > 0.8:
            print(f"  WARNING: {col1} and {col2} are {corr_matrix.iloc[i,j]:.3f} correlated.")
            print(f"           Individual coefficients for these will be unstable.")
            flagged = True

if not flagged:
    print("  None — safe to interpret individual coefficients.")

---
## Step 7: Summary — Linear Regression Rules

In [ ]:
print("=" * 62)
print("DAY 13 SUMMARY: Linear Regression")
print("=" * 62)
print()
print("METRICS")
print("-" * 50)
print("1. R²: fraction of variance explained. 0=baseline, 1=perfect.")
print("2. RMSE: typical error, same units as target, punishes big errors.")
print("3. MAE: typical error, less sensitive to outliers than RMSE.")
print("4. RMSE >> MAE means a few large errors are present.")
print()
print("COEFFICIENTS")
print("-" * 50)
print("5. Each coefficient = effect of that feature, holding others constant.")
print("6. Check correlation between features BEFORE trusting coefficients.")
print("7. Correlation > 0.8-0.9 between features = unstable coefficients")
print("   (multicollinearity), even if predictions stay accurate.")
print()
print("DIAGNOSTICS")
print("-" * 50)
print("8. Residual plot should look like random scatter around 0.")
print("9. Patterns in residuals = model is missing something.")
print("10. Always compare against a 'predict the mean' baseline.")
print()
print("=" * 62)

---
## Practice Exercise

A car price dataset is given below.

Your tasks:
1. Train a `LinearRegression` model using all 3 features
2. Print R², RMSE, and MAE on the test set
3. Read each coefficient and explain what it means in one sentence
4. Check the correlation matrix — are any two features dangerously correlated?
5. Plot predicted vs actual prices

In [ ]:
# Practice dataset — used car prices
np.random.seed(11)
n_practice = 250

car_age_years = np.random.randint(0, 15, n_practice).astype(float)
mileage_km    = np.random.randint(5000, 150000, n_practice).astype(float)
engine_cc     = np.random.choice([1000, 1200, 1500, 1800, 2000], n_practice).astype(float)

# Price (lakhs): newer, lower mileage, bigger engine = more expensive
car_price = (10
              - 0.5 * car_age_years
              - 0.00003 * mileage_km
              + 0.002 * engine_cc
              + np.random.normal(0, 0.8, n_practice))

practice_df = pd.DataFrame({
    'car_age_years': car_age_years,
    'mileage_km': mileage_km,
    'engine_cc': engine_cc,
    'price_lakhs': car_price
})

print("Practice dataset (used car prices):")
print(practice_df.head(8).round(2).to_string(index=False))
print(f"\nShape: {practice_df.shape}")
print()
print("Your tasks:")
print("  1. Train LinearRegression on all 3 features")
print("  2. Print R², RMSE, MAE")
print("  3. Interpret each coefficient in one sentence")
print("  4. Check correlation matrix for multicollinearity")
print("  5. Plot predicted vs actual")

# --- Your solution below ---


In [ ]:
# SOLUTION — try on your own first before looking here

X_car = practice_df[['car_age_years', 'mileage_km', 'engine_cc']].values
y_car = practice_df['price_lakhs'].values

Xtr, Xte, ytr, yte = train_test_split(X_car, y_car, test_size=0.2, random_state=42)

car_model = LinearRegression().fit(Xtr, ytr)
y_car_pred = car_model.predict(Xte)

print("=== Metrics ===")
print(f"R²:   {r2_score(yte, y_car_pred):.4f}")
print(f"RMSE: {mean_squared_error(yte, y_car_pred)**0.5:.4f} lakhs")
print(f"MAE:  {mean_absolute_error(yte, y_car_pred):.4f} lakhs")

print("\n=== Coefficients ===")
names = ['car_age_years', 'mileage_km', 'engine_cc']
for name, coef in zip(names, car_model.coef_):
    print(f"  {name:>15}: {coef:.6f}")

print("\nInterpretation:")
print(f"  Each extra year of age decreases price by about {abs(car_model.coef_[0]):.3f} lakhs.")
print(f"  Each extra km of mileage decreases price by about {abs(car_model.coef_[1]):.6f} lakhs")
print(f"    (or {abs(car_model.coef_[1])*10000:.3f} lakhs per 10,000 km).")
print(f"  Each extra cc of engine size increases price by about {car_model.coef_[2]:.4f} lakhs.")

print("\n=== Correlation Matrix ===")
print(practice_df[names].corr().round(3))
print("\nNo pair exceeds 0.8 — coefficients are safe to interpret individually.")

plt.figure(figsize=(6,5))
plt.scatter(yte, y_car_pred, alpha=0.5, color='seagreen', edgecolor='white')
lims = [min(yte.min(), y_car_pred.min()), max(yte.max(), y_car_pred.max())]
plt.plot(lims, lims, 'r--', linewidth=2)
plt.xlabel('Actual Price (lakhs)')
plt.ylabel('Predicted Price (lakhs)')
plt.title('Used Car Price: Predicted vs Actual')
plt.tight_layout()
plt.savefig('day13_practice.png', dpi=120, bbox_inches='tight')
plt.show()

---
## What's Next

**Day 14: Logistic Regression**  
Why linear regression fails for classification, the sigmoid function, and decision boundaries — using the Titanic dataset.

---
**GitHub repo:** https://github.com/VaishnaviJagtap18/-42-Days-of-ML-Challenge  
**LinkedIn:** Follow for Day 14 tomorrow  
#42DaysOfML #MachineLearning #Python #MLEngineer